# ComplaintIQ - advanced unsupervised representations (`08_unsupervised_advanced`)

`05` clustered raw TF-IDF (ARI ~0.01); `06` improved it with dedup + LSA (ARI ~0.08-0.10) and named
two stronger levers as future work. This notebook **implements** them:

1. **Near-duplicate dedup** with MinHash/LSH (beyond `06`'s exact-match dedup).
2. **Sentence embeddings** (a pretrained transformer) as the text representation, replacing TF-IDF.

Both are scored on the same **`product x issue` yardstick** (ARI / NMI / purity) so the comparison
to `05`/`06` is direct. This runs on a fixed-seed **sample** - the same rationale as `05`: distance
methods and ARI need labels aligned in memory, and `06` established the sample answers the
representation question.

> **Note:** this pulls in heavier libraries (`sentence-transformers` (MinHash dedup uses native Spark MLlib, no extra dep)) that are not in
> the other notebooks' environment; they are added to this notebook's serverless env only.

## How to read this notebook
Spark reads and draws the sample (stratified by `product` so every theme is represented); the
embedding and LSH steps run on that sample. Each representation is clustered with KMeans at
k = #products and scored against the yardstick, head-to-head with the raw-TF-IDF baseline rebuilt
here.

> **Go deeper:**
> - [SBERT pretrained models](https://www.sbert.net/docs/pretrained_models.html): *picking a small, fast sentence-embedding model (e.g. all-MiniLM-L6-v2), ~8 min.*
> - [Spark ML LSH](https://spark.apache.org/docs/latest/ml-features.html#locality-sensitive-hashing): *MinHashLSH and approxSimilarityJoin for distributed near-duplicate detection, ~10 min.*
> - [scikit-learn: adjusted_rand_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html): *why ARI corrects for chance agreement, ~4 min.*

## Setup

In [ ]:
import sys
from pathlib import Path

# Import shared helpers from the local complaintiq package. Walk up from the cwd
# to find the repo's src/ dir, so this works whether the notebook lives in
# notebooks/ or notebooks/appendix/, locally or in a Databricks Git folder.
_here = Path.cwd()
_src = None
for _p in [_here, *_here.parents]:
    if (_p / "src" / "complaintiq").exists():
        _src = str(_p / "src")
        break
if _src and _src not in sys.path:
    sys.path.insert(0, _src)

In [ ]:
from __future__ import annotations
from typing import Any
from complaintiq import RANDOM_STATE, np, pd, plt, sns, print_versions  # shared setup
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

sns.set_theme(style="whitegrid", palette="colorblind", context="notebook", font_scale=1.1)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("setup ok")

---
## 1. Load a representative sample

Spark reads the narrative-only Parquet and draws a fixed-seed sample stratified by `product`; only
the sample goes to the embedding / LSH steps.

In [ ]:
from pathlib import Path

VOLUME_DIR = Path("/Volumes/workspace/complaintiq/data")
data_dir = VOLUME_DIR if VOLUME_DIR.exists() else Path("..") / "data"
nar_path = data_dir / "complaints_narrative_only.parquet"
full_path = data_dir / "complaints.parquet"
src = str(nar_path) if nar_path.exists() else str(full_path)

docs_sdf = spark.read.parquet(src)
if "has_narrative" in docs_sdf.columns and src == str(full_path):
    docs_sdf = docs_sdf.filter(F.col("has_narrative"))
docs_sdf = (
    docs_sdf.select("complaint_text", "product", "issue")
    .dropna(subset=["complaint_text", "product"])
    .filter(F.length("complaint_text") > 0)
)
total = docs_sdf.count()
prods = [r["product"] for r in docs_sdf.select("product").distinct().collect()]
frac = min(1.0, 8_000 / total) if total else 0.0  # smaller sample: the row-wise
# MinHash loop and embedding encode are compute-heavy; 8k is enough for the
# representation comparison (same rationale as 05/06 sampling).
# Keep the sample as a Spark DataFrame (with a stable id) for the native MinHashLSH dedup
# below; also materialize a pandas copy for the sklearn clustering/scoring steps.
sample_sdf = docs_sdf.sampleBy("product", {p: frac for p in prods}, seed=RANDOM_STATE).withColumn(
    "doc_id", F.monotonically_increasing_id()
)
sample = sample_sdf.toPandas().reset_index(drop=True)
print(f"corpus {total:,} -> sample {len(sample):,}")


def yardstick(frame: pd.DataFrame) -> np.ndarray:
    return (
        (frame["product"].astype(str) + " | " + frame["issue"].astype(str))
        .astype("category")
        .cat.codes
    )


def score(X: np.ndarray, labels: np.ndarray, truth: np.ndarray, tag: str) -> dict[str, Any]:
    ari = adjusted_rand_score(truth, labels)
    nmi = normalized_mutual_info_score(truth, labels)
    sil = silhouette_score(X, labels, sample_size=5000, random_state=RANDOM_STATE)
    print(f"{tag:<28} ARI={ari:.4f}  NMI={nmi:.4f}  silhouette={sil:.4f}")
    return {"model": tag, "ari": float(ari), "nmi": float(nmi), "silhouette": float(sil)}


results = []

---
## 2. Baseline reference (raw TF-IDF + KMeans)
Rebuild `05`'s baseline on this sample so the advanced representations are measured against it here.

In [ ]:
base_vec = TfidfVectorizer(max_features=20_000, min_df=5, stop_words="english")
tfidf_baseline = base_vec.fit_transform(sample["complaint_text"])
k = sample["product"].nunique()
results.append(
    score(
        tfidf_baseline,
        KMeans(k, random_state=RANDOM_STATE, n_init=5).fit_predict(tfidf_baseline),
        yardstick(sample),
        f"baseline_tfidf_k{k}",
    )
)

---
## 3. Near-duplicate dedup (MinHash / LSH)

`06` dropped only **exact** duplicates. CFPB narratives include many **near**-duplicates (templated
submissions with small edits). MinHash + LSH finds them without all-pairs comparison: shingle each document, hash the shingles to
a binary vector, MinHash-sign it, and bucket similar signatures. We use Spark MLlib's **native**
`MinHashLSH` so the dedup runs distributed (it scales to the full corpus, unlike a single-node
Python loop): `approxSimilarityJoin` returns near-duplicate pairs, and we drop one representative
per pair before clustering.

> **Go deeper:**
> - [Spark ML MinHashLSH](https://spark.apache.org/docs/latest/ml-features.html#minhash-for-jaccard-distance): *approxSimilarityJoin, numHashTables, and the Jaccard-distance threshold, ~10 min.*
> - [Spark ML LSH overview](https://spark.apache.org/docs/latest/ml-features.html#locality-sensitive-hashing): *how LSH avoids O(n^2) pairwise comparison, ~8 min.*

In [ ]:
# Native Spark MinHashLSH: shingle -> binary HashingTF vector -> MinHash signatures ->
# approxSimilarityJoin to find near-duplicate pairs. Distributed, so this scales past the sample.
from pyspark.ml.feature import Tokenizer, NGram, HashingTF, MinHashLSH

# Character 5-shingles as tokens (via word tokens is coarse for near-dup; use regexp char n-grams).
shingled = (
    sample_sdf.withColumn("norm", F.lower(F.regexp_replace("complaint_text", r"\s+", " ")))
    # 5-char shingles as an array of overlapping substrings
    .withColumn(
        "shingles",
        F.expr(
            "filter(transform(sequence(1, length(norm)-4), i -> substring(norm, i, 5)), x -> x is not null)"
        ),
    )
)
# empty-shingle guard: docs shorter than 5 chars fall back to the whole string
shingled = shingled.withColumn(
    "shingles", F.when(F.size("shingles") > 0, F.col("shingles")).otherwise(F.array("norm"))
)

hashing_tf = HashingTF(inputCol="shingles", outputCol="features", numFeatures=1 << 20, binary=True)
vec = hashing_tf.transform(shingled)
# MinHashLSH needs at least one non-zero feature per row (guaranteed by the fallback above).
minhash_lsh = MinHashLSH(
    inputCol="features", outputCol="hashes", numHashTables=5, seed=RANDOM_STATE
)
mh_model = minhash_lsh.fit(vec)

# Near-duplicate pairs: Jaccard distance < 0.2  <=>  similarity > 0.8. Keep A.doc_id < B.doc_id
# to consider each pair once; the higher doc_id of any near-dup pair is dropped.
pairs = (
    mh_model.approxSimilarityJoin(vec, vec, 0.2, distCol="jdist")
    .select(F.col("datasetA.doc_id").alias("a"), F.col("datasetB.doc_id").alias("b"))
    .filter(F.col("a") < F.col("b"))
)
drop_ids = {r["b"] for r in pairs.select("b").distinct().collect()}  # small set, safe to collect
dedup = sample[~sample["doc_id"].isin(drop_ids)].reset_index(drop=True)
print(
    f"near-dup dedup (Spark MinHashLSH): {len(sample):,} -> {len(dedup):,} kept "
    f"({1 - len(dedup) / len(sample):.1%} were near-duplicates)"
)

In [ ]:
# Cluster the near-dup-deduped set with the same TF-IDF baseline representation, to isolate the
# dedup effect from the representation change in section 4.
tfidf_deduped = base_vec.transform(dedup["complaint_text"])
n_clusters_deduped = dedup["product"].nunique()
results.append(
    score(
        tfidf_deduped,
        KMeans(n_clusters_deduped, random_state=RANDOM_STATE, n_init=5).fit_predict(tfidf_deduped),
        yardstick(dedup),
        f"tfidf_lshdedup_k{n_clusters_deduped}",
    )
)

---
## 4. Sentence embeddings

Replace TF-IDF with a pretrained **sentence-transformer**: each narrative -> one dense ~384-dim
vector that encodes *meaning*, so paraphrases with no shared words land near each other (what TF-IDF
misses). Cluster the embeddings (on the near-dup-deduped set) and score against the yardstick.

> **Go deeper:**
> - [SBERT pretrained models](https://www.sbert.net/docs/pretrained_models.html): *`all-MiniLM-L6-v2` is small and fast; larger models trade speed for quality, ~8 min.*

In [ ]:
from sentence_transformers import SentenceTransformer

# Small, fast model; encodes the deduped sample. normalize -> cosine geometry for KMeans.
model = SentenceTransformer("all-MiniLM-L6-v2")
emb = model.encode(
    dedup["complaint_text"].tolist(),
    batch_size=256,
    show_progress_bar=False,
    normalize_embeddings=True,
)
print("embeddings:", emb.shape)
results.append(
    score(
        emb,
        KMeans(n_clusters_deduped, random_state=RANDOM_STATE, n_init=5).fit_predict(emb),
        yardstick(dedup),
        f"embeddings_k{n_clusters_deduped}",
    )
)

---
## 5. Scoreboard

In [ ]:
board = pd.DataFrame(results).set_index("model")
display(board.round(4))
ax = sns.barplot(x=board["ari"], y=board.index, hue=board.index, palette="colorblind", legend=False)
ax.set_title("ARI vs product x issue: representation levers")
ax.set_xlabel("Adjusted Rand Index (higher = recovers themes better)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

# persist for retrieval outside the run
import json as _json, time

metrics = {
    "notebook": "08_unsupervised_advanced",
    "sample": int(len(sample)),
    "deduped": int(len(dedup)),
    "results": results,
    "ts": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
dbutils.fs.put(
    "/Volumes/workspace/complaintiq/data/metrics_08.json",
    _json.dumps(metrics, indent=2),
    overwrite=True,
)
print("wrote metrics_08.json")

> **What you're seeing:** the raw-TF-IDF baseline, then near-dup dedup, then sentence embeddings -
> each scored on the same yardstick.
>
> **Why it matters:** confirms (with numbers) whether the `06` section 5 levers actually lift ARI/NMI
> over the baseline floor - the honest test of "the unsupervised side is not topped out."

---
## 6. Takeaways

> - **Near-dup dedup** removes templated narratives `06`'s exact-match dedup missed, so clusters are
>   less dominated by artificial groups.
> - **Sentence embeddings** represent meaning, not word overlap, which is what theme clustering wants;
>   compare its ARI to the TF-IDF baseline to see the gain.
> - **Scope:** on a sample (like `05`/`06`), since ARI needs labels in memory and embeddings are
>   compute-heavy; the representation question is sample-appropriate. Full-corpus clustering (hashed
>   TF-IDF) is in `07b`.
> - **Deps:** `sentence-transformers` added to this notebook's serverless env only.